#### LangGraph의 메모리 관리
#### 장기 기억을 위한 메모리 최적화
- SummarizationMiddleware로 대화 요약하기

In [1]:
from langgraph.checkpoint.sqlite import SqliteSaver
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import sqlite3


In [4]:
load_dotenv()

model = ChatOpenAI(model='gpt-5-nano')

# 데이터베이스 연결 및 SqliteSaver 초기화
conn = sqlite3.connect('memory.db', check_same_thread=False) # 기존에 존재하는 동일 기능의 DB가 있는 경우 False로 설정
checkpointer = SqliteSaver(conn)

In [5]:
# 누적 되어진 대화들을 그대로 가져가는 것이 아니라 문제점을 해결할 수 있도록 미들웨어를 장착한다.
from langchain.agents.middleware import SummarizationMiddleware
from langchain.agents import create_agent  # 에이전트 생성

# SummarizationMiddleware : 일정 토근의 크기가 넘어가면 이전 대화의 내용을 요약하는 기능이 탑재 되어 있음
# 1. 요약 미들웨어 설정
summarization_mw = SummarizationMiddleware(
    model=model,
    trigger=[('tokens', 500)],   # 토큰의 max값
    keep=('messages', 5)         # 마지막 5개 메시지만 그대로 유지, 이전 대화는 요약
) 

# 2. Agent 생성(미들웨어 포함)
agent = create_agent(
    model=model,
    checkpointer=checkpointer,      # SqliteSaver 사용
    middleware=[summarization_mw]   # SummarizationMiddleware 미들웨어
)


In [6]:
# 3. 사용자 설정
user_id = 'user_summary'

config = {'configurable': {'thread_id': user_id}}

In [7]:
# 연속 질문 리스트
queries =  [
    "AI의 역사에 대해 간략히 설명해줘.",
    "머신러닝과 딥러닝의 차이점은 뭐야?",
    "트랜스포머 모델의 핵심 원리가 뭐지?",
    "BERT 모델은 어떻게 학습해?",
    "GPT 모델의 특징은 무엇이야?",
    "자연어 처리에서 주로 사용하는 데이터셋에는 어떤 것들이 있어?",
    "AI 윤리 문제에 대해 설명해줄 수 있어?",
    "미래의 AI 기술 발전 방향에 대해 어떻게 생각해?",
    "파이썬의 기초 문법에 대해서 알려줄래?",
    "웹 개발에서 프론트엔드와 백엔드의 차이는 뭐야?",
    "데이터베이스의 정규화란 무엇인가?",
    "클라우드 컴퓨팅의 장점은 뭐야?",
    "컨테이너화 기술에 대해 설명해줘.",
    "마이크로서비스 아키텍처의 특징은 뭐지?",
    "지금까지 우리가 무슨 이야기를 했는지 요약해줄래?"
]

In [8]:
# 4. 모든 질문에 대해서 
for index, query in enumerate(queries, 1):
    print(f'\n[{index}] 사용자: {query}')

    # 에이전트에게 전달
    for chunk in agent.stream(
        {"messages": [{"role": "user", "content": query}]},
        config,
        stream_mode="values"
    ):
        # 각 chunk는 해당 시점의 전체 state를 포함 
        messages = chunk['messages']
        print(f'메세지 갯수: {len(messages)}')

# 마지막 메세지 기준 요약 메세지 확인
for msg in messages:
    if 'summary' in str(msg).lower():
        print(f'\n{msg.content}')



[1] 사용자: AI의 역사에 대해 간략히 설명해줘.
메세지 갯수: 1
메세지 갯수: 2

[2] 사용자: 머신러닝과 딥러닝의 차이점은 뭐야?
메세지 갯수: 3
메세지 갯수: 4

[3] 사용자: 트랜스포머 모델의 핵심 원리가 뭐지?
메세지 갯수: 5
메세지 갯수: 6

[4] 사용자: BERT 모델은 어떻게 학습해?
메세지 갯수: 7
메세지 갯수: 6
메세지 갯수: 7

[5] 사용자: GPT 모델의 특징은 무엇이야?
메세지 갯수: 8
메세지 갯수: 6
메세지 갯수: 7

[6] 사용자: 자연어 처리에서 주로 사용하는 데이터셋에는 어떤 것들이 있어?
메세지 갯수: 8
메세지 갯수: 6
메세지 갯수: 7

[7] 사용자: AI 윤리 문제에 대해 설명해줄 수 있어?
메세지 갯수: 8
메세지 갯수: 6
메세지 갯수: 7

[8] 사용자: 미래의 AI 기술 발전 방향에 대해 어떻게 생각해?
메세지 갯수: 8
메세지 갯수: 6
메세지 갯수: 7

[9] 사용자: 파이썬의 기초 문법에 대해서 알려줄래?
메세지 갯수: 8
메세지 갯수: 6
메세지 갯수: 7

[10] 사용자: 웹 개발에서 프론트엔드와 백엔드의 차이는 뭐야?
메세지 갯수: 8
메세지 갯수: 6
메세지 갯수: 7

[11] 사용자: 데이터베이스의 정규화란 무엇인가?
메세지 갯수: 8
메세지 갯수: 6
메세지 갯수: 7

[12] 사용자: 클라우드 컴퓨팅의 장점은 뭐야?
메세지 갯수: 8
메세지 갯수: 6
메세지 갯수: 7

[13] 사용자: 컨테이너화 기술에 대해 설명해줘.
메세지 갯수: 8
메세지 갯수: 6
메세지 갯수: 7

[14] 사용자: 마이크로서비스 아키텍처의 특징은 뭐지?
메세지 갯수: 8
메세지 갯수: 6
메세지 갯수: 7

[15] 사용자: 지금까지 우리가 무슨 이야기를 했는지 요약해줄래?
메세지 갯수: 8
메세지 갯수: 6
메세지 갯수: 7

Here is a summary of the conversation to date:

## SESSION INTENT

데이터베이스 정규화의 이해를